# Дообучение эмбеддера `multilingual-e5-small` на датасете Adilet Legal QA

Ноутбук выполняет полный цикл дообучения (fine-tuning) модели поиска для НПА РК:

1. **Разбиение данных** на train / val / test **по фрагментам закона** (`context_id`), со стратификацией по законам.
   Все вопросы одного фрагмента попадают только в одну часть — модель не видит тестовые фрагменты при обучении.
2. **Чанкинг** корпуса (фиксированное окно 500 символов, перекрытие 50) и поиск эталонного чанка для каждого вопроса.
3. **Baseline** — исходная модель без дообучения (Dense, BM25, Hybrid) на val и test.
4. **Hard negatives** — сложные отрицательные примеры из выдачи исходной модели.
5. **Дообучение** контрастной функцией потерь `MultipleNegativesRankingLoss`.
6. **Сравнение «до / после»** на тестовой выборке и сохранение результатов.

> Перед запуском: `python prepare_adilet_data.py --lang all` (создаёт `data/raw/*.parquet` и `data/adilet/qa_*_full.json`).
> Нужны пакеты `datasets` и `accelerate` (есть в `requirements.txt`). На CPU полное обучение идёт долго — лучше GPU (Colab / Kaggle).

In [ ]:
# --- Только для Google Colab / Kaggle: раскомментируйте, чтобы скачать проект и данные ---
# !git clone https://github.com/DamirKx/rag.git
# %cd rag
# !pip install -q -r requirements.txt
# !python prepare_adilet_data.py --lang all

## 0. Настройки

In [ ]:
import os
import sys
import json
import re
import random
from collections import defaultdict
from pathlib import Path

# Ноутбук лежит в notebooks/, а модули проекта — в корне репозитория
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from src.chunking import get_fixed_size_chunks
from src.embeddings import EmbeddingModel
from src.retriever import VectorRetriever, BM25Retriever, HybridRetriever, tokenize_legal_text
from src.eval import calculate_mrr, calculate_hit_rate_at_k, calculate_recall_at_k
from prepare_adilet_data import clean_adilet_context
from run_adilet_eval import normalize_text, get_or_compute_embeddings

In [ ]:
LANG = "rus"                                   # "rus" или "kaz"
BASE_MODEL = "intfloat/multilingual-e5-small"
SEED = 42

# Доли train / val / test (считаются по фрагментам закона, а не по вопросам)
SPLIT_RATIOS = (0.8, 0.1, 0.1)

# Чанкинг — как в run_adilet_eval.py
CHUNK_SIZE, CHUNK_OVERLAP = 500, 50
MIN_COVERAGE = 0.5        # мин. доля слов ответа в чанке, чтобы считать его эталонным (см. раздел 3)

# Hard negatives
NEG_SEARCH_DEPTH = 50     # сколько кандидатов достаём из индекса
NEG_POOL = 10             # негатив выбирается случайно из первых NEG_POOL подходящих
NEG_MAX_RATIO = 0.98      # кандидат отбрасывается, если он похож на вопрос почти как эталон (вероятный ложный негатив).
                          # У e5 сходства сжаты (~0.85–0.9), поэтому порог 0.95 из NV-Retriever отсекает больше половины кандидатов

# Обучение
EPOCHS = 3
BATCH_SIZE = 32           # для MNRL чем больше батч, тем больше «бесплатных» негативов
LEARNING_RATE = 2e-5
WARMUP_RATIO = 0.1
MAX_SEQ_LEN = 256         # чанк в 500 символов укладывается в ~150–200 токенов

# Быстрая проверка, что всё работает: маленький корпус и 1 эпоха
SMOKE_TEST = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SPLIT_DIR = Path("data/adilet/splits")
OUTPUT_DIR = Path(f"models/e5-small-adilet-{LANG}")
CHECKPOINT_DIR = Path(f"checkpoints/e5-small-adilet-{LANG}")
CACHE_DIR = "data/adilet/cache"

if SMOKE_TEST:
    EPOCHS = 1
    OUTPUT_DIR = OUTPUT_DIR.with_name(OUTPUT_DIR.name + "-smoke")
    CHECKPOINT_DIR = CHECKPOINT_DIR.with_name(CHECKPOINT_DIR.name + "-smoke")

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print(f"Устройство: {DEVICE} | язык: {LANG} | модель: {BASE_MODEL} | smoke: {SMOKE_TEST}")

## 1. Загрузка данных

In [ ]:
df = pd.read_parquet("data/raw/adilet_legal_qa_kz.parquet")
sub_df = df[df["language"] == LANG].copy().reset_index(drop=True)
sub_df["cleaned_context"] = sub_df["context"].apply(clean_adilet_context)
# Идентификаторы совпадают с prepare_adilet_data.py: ctx_0000, ctx_0001, ...
sub_df["context_id"] = [f"ctx_{i:04d}" for i in range(len(sub_df))]

with open(f"data/adilet/qa_{LANG}_full.json", encoding="utf-8") as f:
    qa_all = json.load(f)

# Проверка, что QA-файл построен по тому же parquet
ctx_law = dict(zip(sub_df["context_id"], sub_df["source_title"]))
assert all(ctx_law[q["context_id"]] == q["source_title"] for q in qa_all), "QA не соответствует parquet — перезапустите prepare_adilet_data.py"

if SMOKE_TEST:
    sub_df = sub_df.sample(n=120, random_state=SEED).sort_index()
    keep = set(sub_df["context_id"])
    qa_all = [q for q in qa_all if q["context_id"] in keep]

print(f"Фрагментов (контекстов): {len(sub_df)} | вопросов: {len(qa_all)}")

## 2. Разбиение train / val / test по фрагментам

Внутри каждого закона фрагменты перемешиваются и делятся в пропорции `SPLIT_RATIOS`.
Так в каждой части представлены все 14 законов, а вопросы к одному фрагменту не «утекают» между частями.
Корпус для поиска при этом остаётся полным: при тестировании модель ищет среди всех чанков всех законов.

In [ ]:
rng = np.random.default_rng(SEED)
split_of_ctx = {}
for law, group in sub_df.groupby("source_title"):
    ids = list(group["context_id"])
    rng.shuffle(ids)
    n_test = max(1, round(len(ids) * SPLIT_RATIOS[2]))
    n_val = max(1, round(len(ids) * SPLIT_RATIOS[1]))
    for j, ctx_id in enumerate(ids):
        split_of_ctx[ctx_id] = "test" if j < n_test else "val" if j < n_test + n_val else "train"

splits = {"train": [], "val": [], "test": []}
for q in qa_all:
    splits[split_of_ctx[q["context_id"]]].append(q)

# Ни один фрагмент не должен попасть в две части
ctx_sets = {name: {q["context_id"] for q in items} for name, items in splits.items()}
assert not (ctx_sets["train"] & ctx_sets["val"]) and not (ctx_sets["train"] & ctx_sets["test"]) and not (ctx_sets["val"] & ctx_sets["test"])

SPLIT_DIR.mkdir(parents=True, exist_ok=True)
if not SMOKE_TEST:
    for name, items in splits.items():
        with open(SPLIT_DIR / f"qa_{LANG}_{name}.json", "w", encoding="utf-8") as f:
            json.dump(items, f, ensure_ascii=False, indent=2)
    print(f"Разбиение сохранено в {SPLIT_DIR}/qa_{LANG}_{{train,val,test}}.json")

stats = pd.DataFrame([
    {"Закон": q["source_title"], "Часть": name}
    for name, items in splits.items() for q in items
]).pivot_table(index="Закон", columns="Часть", aggfunc="size", fill_value=0)[["train", "val", "test"]]
stats.loc["ИТОГО"] = stats.sum()
stats

## 3. Чанкинг корпуса и эталонные чанки

Фрагменты датасета длиной ~1100 токенов не помещаются в лимит e5 (512 токенов), поэтому обучаем и ищем на чанках по 500 символов.
Ответы в датасете — пересказ закона, а не цитата, поэтому точное совпадение подстроки находится редко (~20% вопросов).
Эталонный чанк ищется внутри фрагмента вопроса в три шага:
1. **exact** — чанк содержит начало ответа (`relevant_text`), как в `run_adilet_eval.py`;
2. **overlap** — чанк с наибольшей долей основ слов ответа (первые 6 букв слова), если доля ≥ `MIN_COVERAGE`;
3. **context** — иначе эталоном считаются все чанки фрагмента. Такие вопросы участвуют в оценке, но не в обучении:
   положительный пример должен быть точным.

Из-за шага 2 метрики уровня чанков строже, чем в `run_adilet_eval.py`, и напрямую с ним не сравниваются.

In [ ]:
chunks, chunk_ids, chunk_ctx = [], [], []
for ctx_id, text in zip(sub_df["context_id"], sub_df["cleaned_context"]):
    for s_idx, piece in enumerate(get_fixed_size_chunks(text, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)):
        chunks.append(piece)
        chunk_ids.append(f"{ctx_id}_fix_{s_idx:03d}")
        chunk_ctx.append(ctx_id)

chunk_idx_by_ctx = defaultdict(list)
for i, ctx_id in enumerate(chunk_ctx):
    chunk_idx_by_ctx[ctx_id].append(i)
norm_chunks = [normalize_text(c) for c in chunks]


def stems(text):
    # Грубый стемминг для русского и казахского: первые 6 букв слова
    return {w[:6] for w in tokenize_legal_text(text) if len(w) > 2}


chunk_stems = [stems(c) for c in chunks]


def find_gold_chunks(item):
    # Возвращает (индексы эталонных чанков, способ: exact / overlap / context)
    candidates = chunk_idx_by_ctx[item["context_id"]]
    norm_rel = normalize_text(item.get("relevant_text", ""))
    short_rel = norm_rel[:45]
    matched = [i for i in candidates if short_rel and (short_rel in norm_chunks[i] or norm_rel in norm_chunks[i])]
    if matched:
        return matched, "exact"

    answer_stems = stems(item.get("full_response", ""))
    if answer_stems:
        coverage = [len(answer_stems & chunk_stems[i]) / len(answer_stems) for i in candidates]
        best = int(np.argmax(coverage))
        if coverage[best] >= MIN_COVERAGE:
            return [candidates[best]], "overlap"
    return candidates, "context"


for items in splits.values():
    for q in items:
        q["gold_idx"], q["gold_type"] = find_gold_chunks(q)

print(f"Чанков в корпусе: {len(chunks)}")
pd.DataFrame({
    name: pd.Series([q["gold_type"] for q in items]).value_counts(normalize=True)
    for name, items in splits.items()
}).reindex(["exact", "overlap", "context"]).fillna(0).style.format("{:.1%}")

## 4. Функция оценки

Метрики считаются на уровне чанков (MRR@10, HitRate@1/3/10, Recall@10) и на уровне фрагментов
(`ctx_Hit@k` — найден ли нужный фрагмент закона, сопоставимо со стратегией «контексты целиком»).

In [ ]:
EVAL_K = 10
bm25 = BM25Retriever()
bm25.add_documents(chunks, chunk_ids)
chunk_pos = {cid: i for i, cid in enumerate(chunk_ids)}


def ir_metrics(items, predicted_ids):
    gold = [[chunk_ids[i] for i in q["gold_idx"]] for q in items]
    # Уровень фрагментов: убираем повторы фрагмента, сохраняя порядок выдачи
    pred_ctx = [list(dict.fromkeys(chunk_ctx[chunk_pos[c]] for c in row)) for row in predicted_ids]
    gold_ctx = [[q["context_id"]] for q in items]
    return {
        "MRR@10": calculate_mrr(gold, predicted_ids),
        "Hit@1": calculate_hit_rate_at_k(gold, predicted_ids, k=1),
        "Hit@3": calculate_hit_rate_at_k(gold, predicted_ids, k=3),
        "Hit@10": calculate_hit_rate_at_k(gold, predicted_ids, k=10),
        "Recall@10": calculate_recall_at_k(gold, predicted_ids, k=10),
        "ctx_Hit@1": calculate_hit_rate_at_k(gold_ctx, pred_ctx, k=1),
        "ctx_Hit@3": calculate_hit_rate_at_k(gold_ctx, pred_ctx, k=3),
    }


def build_dense_index(model_name_or_path):
    # Кэш эмбеддингов используется только для моделей с Hub: дообученная модель
    # перезаписывается при каждом запуске, и старые векторы были бы неверны.
    embedder = EmbeddingModel(str(model_name_or_path), device=DEVICE)
    safe_name = re.sub(r"[^a-zA-Z0-9_\-]", "_", str(model_name_or_path))
    chunk_emb = get_or_compute_embeddings(
        embedder, chunks, CACHE_DIR,
        cache_key=f"{safe_name}_{LANG}_fixed_{CHUNK_SIZE}_{CHUNK_OVERLAP}",
        use_cache=not Path(model_name_or_path).exists(),
    )
    index = VectorRetriever(embedding_dim=chunk_emb.shape[1])
    index.add_documents(chunk_emb, chunks, chunk_ids)
    return embedder, index


def evaluate(embedder, index, items):
    queries = [q["query"] for q in items]
    q_emb = embedder.encode_queries(queries)
    _, idx = index.search(q_emb, k=EVAL_K)
    dense_pred = [[chunk_ids[i] for i in row if i >= 0] for row in idx]
    hybrid_pred = HybridRetriever(index, bm25).batch_search(queries, q_emb, k=EVAL_K)
    return {"dense": ir_metrics(items, dense_pred), "hybrid": ir_metrics(items, hybrid_pred)}, dense_pred


def evaluate_bm25(items):
    return ir_metrics(items, bm25.batch_search([q["query"] for q in items], k=EVAL_K))

## 5. Baseline: модель без дообучения

In [ ]:
base_embedder, base_index = build_dense_index(BASE_MODEL)

results = {"bm25": {}, "base": {}, "finetuned": {}}
base_dense_pred = {}
for name in ["val", "test"]:
    results["bm25"][name] = evaluate_bm25(splits[name])
    results["base"][name], base_dense_pred[name] = evaluate(base_embedder, base_index, splits[name])

pd.DataFrame({
    "BM25": results["bm25"]["test"],
    "Dense (base)": results["base"]["test"]["dense"],
    "Hybrid (base)": results["base"]["test"]["hybrid"],
}).T.round(4)

## 6. Обучающие тройки и hard negatives

Для каждого train-вопроса: `(вопрос, эталонный чанк, сложный негатив)`.
Негатив — чанк, который исходная модель считает похожим на вопрос, но который:
- взят из **другого** фрагмента (соседние чанки того же фрагмента могут содержать часть ответа);
- не дублирует текст эталона;
- похож на вопрос заметно меньше эталона (`sim < NEG_MAX_RATIO * sim_pos`) — отсекает вероятные ложные негативы (подход NV-Retriever).

In [ ]:
train_items = [q for q in splits["train"] if q["gold_type"] != "context"]
train_q_emb = base_embedder.encode_queries([q["query"] for q in train_items])
neg_scores, neg_idx = base_index.search(train_q_emb, k=NEG_SEARCH_DEPTH)

chunk_idx_by_law = defaultdict(list)
for i, ctx_id in enumerate(chunk_ctx):
    chunk_idx_by_law[ctx_law[ctx_id]].append(i)

triples = []
fallback = 0
for q, q_vec, scores, row in zip(train_items, train_q_emb, neg_scores, neg_idx):
    pos = q["gold_idx"][0]
    pos_score = float(q_vec @ base_index.index.reconstruct(pos))
    candidates = [
        i for i, s in zip(row, scores)
        if i >= 0 and chunk_ctx[i] != q["context_id"] and chunks[i] != chunks[pos] and s < NEG_MAX_RATIO * pos_score
    ][:NEG_POOL]
    if candidates:
        neg = int(rng.choice(candidates))
    else:
        # Подходящих кандидатов нет: берём случайный чанк того же закона из другого фрагмента
        fallback += 1
        same_law = [i for i in chunk_idx_by_law[q["source_title"]] if chunk_ctx[i] != q["context_id"]]
        pool = same_law or [i for i in range(len(chunks)) if chunk_ctx[i] != q["context_id"]]
        neg = int(rng.choice(pool))
    triples.append({"anchor": q["query"], "positive": chunks[pos], "negative": chunks[neg]})

print(f"Обучающих троек: {len(triples)} (случайный негатив: {fallback})")
example = triples[0]
print("\nВопрос:  ", example["anchor"])
print("Эталон:  ", example["positive"][:200], "...")
print("Негатив: ", example["negative"][:200], "...")

## 7. Дообучение

- Функция потерь `MultipleNegativesRankingLoss`: для каждого вопроса эталон должен быть ближе, чем hard negative
  и чанки остальных вопросов батча (in-batch negatives).
- `BatchSamplers.NO_DUPLICATES` не кладёт в один батч одинаковые тексты: у одного чанка бывает несколько вопросов,
  и без этого эталон одного вопроса стал бы ложным негативом для другого.
- Префиксы `query: ` / `passage: ` добавляются вручную, как при инференсе в `src/embeddings.py`.
- После каждой эпохи считается MRR@10 на val; в конце загружается лучшая эпоха.

In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.sentence_transformer import losses
from sentence_transformers.sentence_transformer.evaluation import InformationRetrievalEvaluator
from sentence_transformers.sentence_transformer.training_args import BatchSamplers

model = SentenceTransformer(BASE_MODEL, device=DEVICE)
model.max_seq_length = MAX_SEQ_LEN

train_dataset = Dataset.from_dict({
    "anchor": [f"query: {t['anchor']}" for t in triples],
    "positive": [f"passage: {t['positive']}" for t in triples],
    "negative": [f"passage: {t['negative']}" for t in triples],
}).shuffle(seed=SEED)

val_evaluator = InformationRetrievalEvaluator(
    queries={q["id"]: f"query: {q['query']}" for q in splits["val"]},
    corpus={cid: f"passage: {text}" for cid, text in zip(chunk_ids, chunks)},
    relevant_docs={q["id"]: {chunk_ids[i] for i in q["gold_idx"]} for q in splits["val"]},
    name="val",
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[1, 3, 10],
    precision_recall_at_k=[10],
    batch_size=64,
)

loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_RATIO,  # в Transformers v5 дробное значение означает долю шагов
    fp16=DEVICE == "cuda",
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_val_cosine_mrr@10",
    greater_is_better=True,
    logging_steps=20,
    seed=SEED,
    report_to="none",
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    loss=loss,
    evaluator=val_evaluator,
)
trainer.train()

model.save(str(OUTPUT_DIR))
print(f"Дообученная модель сохранена в {OUTPUT_DIR}")

In [ ]:
# Динамика обучения: loss и MRR@10 на val по эпохам
history = pd.DataFrame(trainer.state.log_history)
cols = [c for c in ["epoch", "loss", "eval_val_cosine_mrr@10", "eval_val_cosine_accuracy@1", "eval_val_cosine_accuracy@3"] if c in history]
history[cols].groupby("epoch").last().round(4)

## 8. Сравнение «до / после» на тестовой выборке

In [ ]:
ft_embedder, ft_index = build_dense_index(OUTPUT_DIR)
ft_dense_pred = {}
for name in ["val", "test"]:
    results["finetuned"][name], ft_dense_pred[name] = evaluate(ft_embedder, ft_index, splits[name])

comparison = pd.DataFrame({
    "BM25": results["bm25"]["test"],
    "Dense (base)": results["base"]["test"]["dense"],
    "Dense (fine-tuned)": results["finetuned"]["test"]["dense"],
    "Hybrid (base)": results["base"]["test"]["hybrid"],
    "Hybrid (fine-tuned)": results["finetuned"]["test"]["hybrid"],
}).T
comparison.round(4)

In [ ]:
# Примеры вопросов, где исходная модель промахнулась в топ-3, а дообученная нашла эталон на 1-м месте
test_items = splits["test"]
improved = [
    (q, b, f) for q, b, f in zip(test_items, base_dense_pred["test"], ft_dense_pred["test"])
    if not {chunk_ids[i] for i in q["gold_idx"]} & set(b[:3]) and f[:1] and chunk_pos[f[0]] in q["gold_idx"]
]
print(f"Улучшено вопросов: {len(improved)} из {len(test_items)}\n")
for q, b, f in improved[:5]:
    print(f"[{q['source_title']}] {q['query']}")
    print(f"  base топ-1:       {chunks[chunk_pos[b[0]]][:150]}...")
    print(f"  fine-tuned топ-1: {chunks[chunk_pos[f[0]]][:150]}...\n")

In [ ]:
if not SMOKE_TEST:
    report = {
        "language": LANG,
        "base_model": BASE_MODEL,
        "finetuned_model": str(OUTPUT_DIR),
        "seed": SEED,
        "split_ratios": SPLIT_RATIOS,
        "split_sizes": {name: len(items) for name, items in splits.items()},
        "train_triples": len(triples),
        "hyperparams": {
            "epochs": EPOCHS, "batch_size": BATCH_SIZE, "lr": LEARNING_RATE,
            "warmup_ratio": WARMUP_RATIO, "max_seq_len": MAX_SEQ_LEN,
            "chunk_size": CHUNK_SIZE, "chunk_overlap": CHUNK_OVERLAP,
        },
        "results": results,
    }
    report_path = Path(f"data/adilet/finetune_results_{LANG}.json")
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"Отчёт сохранён: {report_path}")

## Дальнейшие шаги

- Проверить модель в интерактивном поиске: `python ask.py --model models/e5-small-adilet-rus`.
- Запустить ноутбук с `LANG = "kaz"` и с 2–3 разными `SEED`, чтобы показать устойчивость прироста.
- Сравнить с `intfloat/multilingual-e5-base` (поменять `BASE_MODEL`, уменьшить `BATCH_SIZE` при нехватке памяти).